In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Herramientas de Clasificación
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import accuracy_score, classification_report 

# Cargar dataset
df = pd.read_csv('../data/processed/movies_listo_para_modelo.csv')
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=['roi', 'budget', 'temporada'])

print(f" Datos cargados: {df.shape[0]} películas.")

In [ ]:
# 1. Crear la nueva variable objetivo binaria (Éxito vs Fracaso)
df['es_rentable'] = (df['roi'] > 0).astype(int)

# 2. Definir el nuevo objetivo (y) y mantener tus features (X)
y = df['es_rentable']
features = ['budget', 'popularity', 'temporada', 'es_drama', 'es_comedia', 'es_thriller']
X = df[features]

# 3. Dividir en Entrenamiento (80%) y Prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f" Entrenamiento: {X_train.shape[0]} |  Prueba: {X_test.shape[0]}")

In [ ]:
num_features = ['budget', 'popularity', 'es_drama', 'es_comedia', 'es_thriller']
cat_features = ['temporada']

# Preprocesamiento
preprocesador = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ])

# Ensamblar el Pipeline con Clasificador
modelo_pipeline_clf = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('algoritmo', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Entrenar el modelo
modelo_pipeline_clf.fit(X_train, y_train)

print(" Modelo de Clasificación entrenado exitosamente.")

In [ ]:
y_pred_clf = modelo_pipeline_clf.predict(X_test)

# Calcular Exactitud (Accuracy)
exactitud = accuracy_score(y_test, y_pred_clf)

print("---  RESULTADOS DE CLASIFICACIÓN ---")
print(f"Exactitud del Modelo (Accuracy): {exactitud * 100:.2f}%")
print("\nReporte Detallado:")
print(classification_report(y_test, y_pred_clf))

In [ ]:
cat_encoder = modelo_pipeline_clf.named_steps['preprocesador'].transformers_[1][1]
cat_nombres = cat_encoder.get_feature_names_out(cat_features)
nombres_finales = num_features + list(cat_nombres)

importancias = modelo_pipeline_clf.named_steps['algoritmo'].feature_importances_

df_importancia_clf = pd.DataFrame({
    'Variable': nombres_finales,
    'Importancia': importancias
}).sort_values(by='Importancia', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importancia', y='Variable', data=df_importancia_clf, palette='viridis')
plt.title('Variables clave para predecir Éxito o Fracaso (Clasificación)', fontsize=14)
plt.xlabel('Nivel de Importancia (0 a 1)')
plt.ylabel('Característica')
plt.tight_layout()
plt.savefig('../outputs/figures/importancia_clasificacion.png')
plt.show()